![image.png](https://i.imgur.com/4fN73lZ.png)

## Setup

We install with **uv** and use `imageio` for GIF rendering (consistent with the other RL labs).

In [ ]:
# Fast dependency install with uv (https://docs.astral.sh/uv).
# Bootstraps uv via pip, then installs into the current kernel/venv. If you already
# run from the project's .venv (created with `uv sync`), this is essentially a no-op.
import sys, os
%pip install -q uv
_target = "" if (sys.prefix != sys.base_prefix or os.environ.get("VIRTUAL_ENV")) else "--system"
!uv pip install -q {_target} --python "{sys.executable}" gymnasium flappy_bird_gymnasium imageio matplotlib torch

# Content

In this demo, we will build a custom PPO model implementation using pytorch and then train that implementation on flappy bird game.

for the environment, we will use flappy_bird_gymnasium environment. You can read more about this environment [here](https://github.com/markub3327/flappy-bird-gymnasium)

## PPO

> **Note:** This is a complete **PPO-Clip** implementation: each rollout is reused for `k_epochs` gradient steps, and the probability ratio compares the *current* policy to the policy that collected the data (`old_log_probs`) — that ratio is what the clipping acts on. It uses the same `ActorCritic` / `PPOAgent` as the LunarLander PPO lab; here the rollout is a single episode.

PPO is motivated by the same question as TRPO: how can we take the biggest possible improvement step on a policy using the data we currently have, without stepping so far that we accidentally cause performance collapse? Where TRPO tries to solve this problem with a complex second-order method, PPO is a family of first-order methods that use a few other tricks to keep new policies close to old. PPO methods are significantly simpler to implement, and empirically seem to perform at least as well as TRPO.

There are two primary variants of PPO: PPO-Penalty and PPO-Clip.

**PPO-Penalty** approximately solves a KL-constrained update like TRPO, but penalizes the KL-divergence in the objective function instead of making it a hard constraint, and automatically adjusts the penalty coefficient over the course of training so that it's scaled appropriately.

**PPO-Clip** doesn't have a KL-divergence term in the objective and doesn’t have a constraint at all. Instead relies on specialized clipping in the objective function to remove incentives for the new policy to get far from the old policy.

Read more [here](https://spinningup.openai.com/en/latest/algorithms/ppo.html).

Here, we'll focus only on PPO-Clip.

![ppo_2.png](https://i.imgur.com/VCKH7yN.png)

[Image Source](http://rail.eecs.berkeley.edu/deeprlcourse-fa17/)

In [ ]:
import gymnasium as gym
import flappy_bird_gymnasium
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## Let's define our model

In [ ]:
# Actor-critic network: a shared body with a policy head (actor) and a value head (critic).
class ActorCritic(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, 32)
        self.actor = nn.Linear(32, output_dim)   # action logits
        self.critic = nn.Linear(32, 1)           # state value V(s)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        return self.actor(x), self.critic(x)


# A rollout simply stores the transitions collected under the current policy for one update.
class Rollout:
    def __init__(self):
        self.clear()

    def clear(self):
        self.states, self.actions, self.rewards, self.dones, self.log_probs = [], [], [], [], []

    def add(self, state, action, reward, done, log_prob):
        self.states.append(state)
        self.actions.append(action)
        self.rewards.append(reward)
        self.dones.append(done)
        self.log_probs.append(log_prob)

In [ ]:
class PPOAgent:
    def __init__(self, input_dim, output_dim, lr=1e-3, gamma=0.99, epsilon=0.2,
                 k_epochs=4, device="cpu"):
        self.actor_critic = ActorCritic(input_dim, output_dim).to(device)
        self.optimizer = optim.Adam(self.actor_critic.parameters(), lr=lr)
        self.gamma = gamma
        self.epsilon = epsilon
        self.k_epochs = k_epochs
        self.device = device

    def select_action(self, state):
        """Sample an action under the current policy; return (action, log_prob).
        The log_prob is kept as the OLD log-prob for the PPO ratio."""
        state = torch.tensor(np.asarray(state), dtype=torch.float32, device=self.device)
        with torch.no_grad():
            logits, _ = self.actor_critic(state)
            dist = Categorical(F.softmax(logits, dim=-1))
            action = dist.sample()
        return action.item(), dist.log_prob(action)

    def compute_returns(self, rewards, dones):
        """Discounted returns over the rollout, reset at episode boundaries (dones)."""
        returns, R = [], 0.0
        for r, done in zip(reversed(rewards), reversed(dones)):
            if done:
                R = 0.0
            R = r + self.gamma * R
            returns.insert(0, R)
        return torch.tensor(returns, dtype=torch.float32, device=self.device)

    def compute_advantages(self, states, returns):
        """Advantage = return - V(s) (the critic as a baseline), normalised for stable gradients."""
        with torch.no_grad():
            states_t = torch.tensor(np.array(states), dtype=torch.float32, device=self.device)
            values = self.actor_critic(states_t)[1].squeeze(-1)
        advantages = returns - values
        return (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    def ppo_loss(self, states, actions, returns, advantages, old_log_probs):
        """The PPO-Clip loss for one pass over the rollout:
        clipped policy loss + value loss - entropy bonus."""
        logits, values = self.actor_critic(states)
        dist = Categorical(F.softmax(logits, dim=-1))
        log_probs = dist.log_prob(actions)

        ratio = torch.exp(log_probs - old_log_probs)             # pi_new / pi_old
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1 - self.epsilon, 1 + self.epsilon) * advantages
        policy_loss = -torch.min(surr1, surr2).mean()

        value_loss = F.smooth_l1_loss(values.squeeze(-1), returns)
        entropy = dist.entropy().mean()
        return policy_loss + 0.5 * value_loss - 0.01 * entropy

    def update(self, states, actions, returns, advantages, old_log_probs):
        """Reuse the rollout for k_epochs of gradient steps on the PPO loss."""
        states = torch.tensor(np.array(states), dtype=torch.float32, device=self.device)
        actions = torch.tensor(actions, dtype=torch.int64, device=self.device)
        old_log_probs = torch.stack(old_log_probs).detach().to(self.device)
        for _ in range(self.k_epochs):
            loss = self.ppo_loss(states, actions, returns, advantages, old_log_probs)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

## Initialize the environment and model

In [ ]:
env = gym.make("FlappyBird-v0", render_mode="rgb_array", use_lidar=False)

In [ ]:
input_dim = env.observation_space.shape[0]
output_dim = env.action_space.n
num_episodes = 10000
max_steps = 1000

lr = 1e-3
gamma = 0.99
epsilon = 0.2
k_epochs = 4

In [ ]:
agent = PPOAgent(input_dim, output_dim, lr, gamma, epsilon, k_epochs, device=device)

## Training the model

In [ ]:
scores = []
rollout = Rollout()
for episode in range(num_episodes):
    state = env.reset()[0]
    rollout.clear()   # Flappy updates after every episode (a one-episode rollout)
    for step in range(max_steps):
        action, log_prob = agent.select_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        rollout.add(state, action, reward, terminated or truncated, log_prob)
        state = next_state
        if terminated or truncated:
            break
    scores.append(sum(rollout.rewards))

    returns = agent.compute_returns(rollout.rewards, rollout.dones)
    advantages = agent.compute_advantages(rollout.states, returns)
    agent.update(rollout.states, rollout.actions, returns, advantages, rollout.log_probs)

    if (episode + 1) % 50 == 0:
        print(f"Episode {episode + 1:5d} | avg reward (last 50): {np.mean(scores[-50:]):7.2f}")

env.close()

## Visualizing model's performance

In [ ]:
# Roll out the trained policy and save it as a GIF (Gymnasium-native)
import os
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")  # headless rendering (e.g. Colab)
import imageio.v2 as imageio
from IPython.display import Image, display

os.makedirs("video", exist_ok=True)
eval_env = gym.make("FlappyBird-v0", render_mode="rgb_array", use_lidar=False)
state = eval_env.reset()[0]
frames = []
for _ in range(2000):
    frames.append(eval_env.render())
    action, _ = agent.select_action(state)
    state, reward, terminated, truncated, _ = eval_env.step(action)
    if terminated or truncated:
        break
eval_env.close()
imageio.mimsave("video/flappy.gif", frames, fps=24)

In [ ]:
display(Image(filename="video/flappy.gif"))